In [15]:
import pandas as pd

In [16]:
# read file
df = pd.read_csv(
    "../data/raw/listings.csv", skipinitialspace=True, na_values=["None"]
)

In [17]:
# drop duplicates
print(df.duplicated().sum())

df = df.drop_duplicates()
# check the number. should be zero
print(df.duplicated().sum())

651
0


In [18]:
# fill swiming pool values with 0 for missing values
df["swimming_pool"] = df["swimming_pool"].fillna(0)

In [19]:
# fill garden area missing values with zero.
df["garden"] = df["garden"].fillna(0)

In [20]:
# fill facades number missing values with zero.
# df['num_facades'] = df['num_facades'].fillna(0)
# fill with median. better?

df['num_facades'] = df['num_facades'].fillna(df['num_facades'])

print(df['state_of_building'].mean())

TypeError: unsupported operand type(s) for +: 'int' and 'str'

In [ ]:
# only fill garden_area_m2 with 0 if there is no garden
df.loc[df["garden"] == 0, "garden_area_m2"] = df.loc[
    df["garden"] == 0, "garden_area_m2"
].fillna(0)

In [ ]:
# replace missing values in fully_equiped_kitchen with 0
df["fully_equipped_kitchen"] = df["fully_equipped_kitchen"].fillna(0)

In [ ]:
# fill terrace area missing values with zero.
df["terrace"] = df["terrace"].fillna(0)

In [ ]:
# only fill terrace_area_m2 with 0 if ther is no terrace
df.loc[df["terrace"] == 0, "terrace_area_m2"] = df.loc[
    df["terrace"] == 0, "terrace_area_m2"
].fillna(0)

In [ ]:
# clean rows with living area less than 10m2
df = df[df["living_area_m2"] > 10]

In [ ]:
# clean price les than 10000
df = df[df["price_eur"] > 10000]

NameError: name 'df' is not defined

In [ ]:
# remove garden area more than 5000 m2. data error or unusual property. keep the null values
df = df[
    (df["garden_area_m2"] <= 5000) | (df["garden_area_m2"].isna())
]  # only 137 properties

In [ ]:
# remove living area more than 1000 m2. data error or unusual property. keep the null values
df = df[
    (df["living_area_m2"] <= 1000) | (df["living_area_m2"].isna())
]  # only 57 properties

In [ ]:
# remove land surface area less than 10m2 and more than 10k m2. keep the null values
df = df[
    ((df["land_surface_m2"] >= 10) & (df["land_surface_m2"] <= 10000))
    | (df["land_surface_m2"].isna())
]  # only 111 properties

In [ ]:
# drop rows without price
df = df.dropna(subset=["price_eur"])

In [ ]:
# remove houses with price less than 20k. most likely errors.
df = df[df["price_eur"] >= 20000]  # only 6 property

In [ ]:
# create price_per_sqm
df["price_per_m2"] = (df["price_eur"] / df["living_area_m2"]).round(2)

In [ ]:
# Create postal codes and assign to each region
df["postal_code"] = (
    df["locality"].str.extract(r"(\d{4})").astype(int)
)  # take only the first 4 digit from locality and turn it into int


# assign postal codes based on region
def get_region(postal_code):
    if 1000 <= postal_code <= 1299:
        return "Brussels"
    elif (1300 <= postal_code <= 1499) or (4000 <= postal_code <= 7999):
        return "Wallonia"
    else:
        return "Flanders"


df["region"] = df["postal_code"].apply(get_region)

In [ ]:
# Create outliers True or False for prices more than 2m and less than 4 rooms
df["outlier_flag"] = (df["price_eur"] > 2000000) & (df["num_rooms"] < 4)

In [ ]:
# create a new column with target encoding. assign avg price by locality
df["avg_price_locality"] = (
    df.groupby("locality")["price_eur"].transform("mean").round(1)
)

In [22]:
df["state_of_building"].unique()  # check unique values
state_order = {  # assign values based on buildind state
    "To demolish": 1,
    "To restore": 2,
    "To renovate": 3,
    "To be renovated": 4,
    "Normal": 5,
    "Under construction": 6,
    "Fully renovated": 7,
    "Excellent": 8,
    "New": 9,
}
df["building_state_encoded"] = df["state_of_building"].map(state_order)

print(df['building_state_encoded'].mean())

6.148511383537653


In [ ]:
# reset the index after cleaning
df = df.reset_index(drop=True)

In [ ]:
# save to a new csv
df.to_csv("../data/processed/cleaned_listings.csv", index=False)